# Lab 19: Tree-Based Models — Random Forests
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 30 min Core + 15 min Extension + SHAP Deep Dive

---

**Format:** This lab contains **deliberately flawed code and analysis**. Your job:
1. Run the code
2. Identify what is wrong (not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Extend the corrected analysis

**Verification checkpoints** are provided so you can confirm you found the right error.

---

In [3]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 1: Import libraries and load data
# -----------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

## Part 1: Find the Bug — Model Comparison (10 min)

The following code trains three models and reports their performance.
**Something is wrong with how the comparison is set up.** Find it, fix it, explain.

In [4]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains deliberate error)
# Step 2: Model comparison — find the bug
# -----------------------------------------------------------

tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

# BUG IS HERE: RF is evaluated on TRAINING data, not test data
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

print('=== Model Comparison ===')
print(f"Single Tree  \u2014 R\u00b2: {r2_score(y_test, tree.predict(X_test)):.4f}")
print(f"Ridge        \u2014 R\u00b2: {r2_score(y_test, ridge.predict(X_test)):.4f}")
print(f"Random Forest \u2014 R\u00b2: {r2_score(y_train, rf.predict(X_train)):.4f}")  # \u2190 WRONG: using training set
print()
print('Conclusion: Random Forest achieves R\u00b2 > 0.97! Far superior to alternatives.')

=== Model Comparison ===
Single Tree  — R²: 0.6221
Ridge        — R²: 0.5759
Random Forest — R²: 0.9736

Conclusion: Random Forest achieves R² > 0.97! Far superior to alternatives.


### YOUR DIAGNOSIS

1. **What is wrong?** (identify the specific line and error type)
- The Random Forest Model is being tested on the training data whiltst the other models are being tested with the test data. This makes the R-squared comparisons misleading because the Single Tree and Ridge models are being tested with data the model has not seen before.
2. **Why is this dangerous?** (what misleading conclusion does it lead to?)
- This can made a model seem like it has significant prediction accuracy. Running a cross validation test and using training data to test can hide bias within your model. You need new data to understand how well your model runs with data its not trained on.
3. **Fix the code below** and report the correct R²

**Verification checkpoint:** After fixing, the RF Test R² should be between 0.78 and 0.83. If you get >0.95, you haven't found the bug.

4. **Which chapter concept does this error violate?** (hint: Ch 15)
This fails the concept of bias variance trade off. Having test data reduces the liklihood of overfitting, when you overfit the model the data can often follow your sample trend rather than the true population trend, which is why we need untrained data to understand the prediction accuracy.

In [5]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Fix the model comparison bug from Part 1
# -----------------------------------------------------------
# YOUR FIX HERE

print(f"Random Forest — R²: {r2_score(y_test, rf.predict(X_test)):.4f}")


Random Forest — R²: 0.8051


## Part 2: Find the Methodological Flaw — Feature Importance (10 min)

The following analysis uses feature importance to make a **causal claim**.
The code runs correctly. The methodology is wrong. Find the flaw.

In [6]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains methodological flaw)
# Step 3: Feature importance with flawed causal reasoning
# -----------------------------------------------------------

rf_correct = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
rf_correct.fit(X_train, y_train)

importance = pd.Series(rf_correct.feature_importances_, index=X.columns).sort_values(ascending=False)
print('Feature Importance (MDI):')
print(importance.round(4))
print()
print('POLICY RECOMMENDATION:')
print(f'The top predictor is {importance.index[0]} (importance = {importance.iloc[0]:.3f}).')
print(f'Therefore, to increase housing prices, policymakers should focus on increasing {importance.index[0]}.')
print(f'The second most important lever is {importance.index[1]}.')

Feature Importance (MDI):
MedInc        0.5259
AveOccup      0.1381
Latitude      0.0886
Longitude     0.0883
HouseAge      0.0544
AveRooms      0.0444
Population    0.0307
AveBedrms     0.0296
dtype: float64

POLICY RECOMMENDATION:
The top predictor is MedInc (importance = 0.526).
Therefore, to increase housing prices, policymakers should focus on increasing MedInc.
The second most important lever is AveOccup.


### YOUR DIAGNOSIS

1. **What is the methodological flaw?** (the code is correct — the reasoning is wrong)   The MDI will tell us the variables that had the largest weights within our prediction however we cant draw causality from this method.
2. **Why can't we use MDI for policy recommendations?** (connect to Ch 10 DAGs and Ch 15 prediction vs. explanation) MDI tells us which variable reduced impurity the most; it wont be able to give a causal analysis. There are many factors correlated with MedInc  (school quality, safety and proximity to work) that are also related to to housing prices that could create alot of upward bias in the model.
3. **What would you need to make a causal claim?** (hint: Ch 24 DML)   The Double Machine Learning Method helps us understand causality because it cancels out the noise from cofounding variables. By taking the residuals from the the other features on MedInc to regress on the residuals from the the other features and housing, we are canceling out the bias that would be contributed by the cofounders. This isolates the variables enough to diagnose causality.

4. **Bonus:** MDI has a known statistical bias. What is it, and what alternative would you use?
MDI is very sensitive to features that can have many unique variables. So any continuous variable has many more oppertunities to split than a low cardinality feature like number of bedrooms. Because of this the model could find impurity reduction in a feature that has many chances to split up and give weight to a completely meaningless feature.

**Verification checkpoint:** Your diagnosis should mention at least: (a) prediction ≠ causation, (b) confounding/omitted variables, (c) MDI bias toward high-cardinality features.

In [7]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Run permutation importance and write a proper (non-causal)
# interpretation of the results
# -----------------------------------------------------------

# YOUR CORRECTED ANALYSIS HERE

perm_result = permutation_importance(rf_correct, X_test, y_test, n_repeats=10, random_state=42)
perm_importance = pd.Series(perm_result.importances_mean, index=X.columns).sort_values(ascending=False)
print("Permutation Importance (unbiased):")
print(perm_importance.round(4))
print("\nNOTE: These features PREDICT housing prices. For CAUSAL claims, use DML (Ch 24).")

Permutation Importance (unbiased):
MedInc        0.7347
Latitude      0.4429
Longitude     0.3352
AveOccup      0.2036
HouseAge      0.0721
AveRooms      0.0271
AveBedrms     0.0095
Population    0.0087
dtype: float64

NOTE: These features PREDICT housing prices. For CAUSAL claims, use DML (Ch 24).


## Part 3: Hyperparameter Tuning + XGBoost Comparison (10 min)

Tune the RF, then compare against XGBoost (gradient boosting).

In [8]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Tune RF with GridSearchCV and compare with GBR
# -----------------------------------------------------------
# I Shrunk the param_grid and CV because the runtime for this cell was > 15min
'''
original:
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [10, 20, None],
    'max_features': ['sqrt', 0.5],
}
'''
# original:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, None],
    'max_features': ['sqrt'],
}

# 1. GridSearchCV on RandomForestRegressor
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=3, scoring='neg_mean_squared_error', n_jobs=-1
)
grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_


# 2. Fit GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1)

gbr = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
gbr.fit(X_train, y_train)

# 3. Compare Test RMSE and R\u00b2 for: Ridge, RF (default), RF (tuned), GBR

for name, model in [('Ridge', ridge), ('RF (default)', rf), ('RF (tuned)', best_rf), ('GBR', gbr)]:
    rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    r2 = r2_score(y_test, model.predict(X_test))
    print(f"{name:20s} — RMSE: {rmse:.4f}, R²: {r2:.4f}")
# 4. Which model wins? By how much? Is the difference practically significant?


Ridge                — RMSE: 0.7455, R²: 0.5759
RF (default)         — RMSE: 0.5053, R²: 0.8051
RF (tuned)           — RMSE: 0.4939, R²: 0.8138
GBR                  — RMSE: 0.4736, R²: 0.8288


---

## Extension: SHAP Analysis (5200 depth — 15 min)

Use SHAP to explain individual predictions. Compare MDI ranking vs. SHAP ranking.

In [9]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 4: SHAP setup and TreeExplainer
# -----------------------------------------------------------

# Install SHAP if needed
# !pip install shap
import shap

# Create SHAP explainer for the tuned RF
# explainer = shap.TreeExplainer(best_rf)  # use your tuned RF from Part 3
# shap_values = explainer.shap_values(X_test)

# 1. Waterfall plot for 3 observations: one high-value, one low-value, one surprising
# shap.plots.waterfall(shap.Explanation(values=shap_values[0], base_values=explainer.expected_value, data=X_test.iloc[0]))

# 2. Beeswarm plot (global view)
# shap.plots.beeswarm(shap.Explanation(values=shap_values, base_values=explainer.expected_value, data=X_test))

# 3. Compare MDI ranking vs SHAP ranking \u2014 do they agree? Where do they diverge?

### SHAP Interpretation (write as a .py module)

Create a reusable `shap_analysis.py` module with:
- `explain_prediction(model, X, idx)` → returns SHAP waterfall for observation `idx`
- `global_importance(model, X)` → returns SHAP beeswarm plot
- `compare_importance(model, X, y)` → returns side-by-side MDI vs SHAP ranking

Include docstrings and type hints. This is a portfolio artifact.

---
## AI-Assisted Expansion: SHAP Dashboard + Reusable Module

**The Generative AI Policy: Foundations First, Expansion Second.** You have now established manual mastery over decision trees, random forests, hyperparameter tuning, feature importance, and SHAP explanations. You are now authorized to operate under the "Co-Pilot Rule."

### Your Expansion Task (5200 — Advanced)
Build TWO artifacts:

**Artifact 1: `src/shap_utils.py` module** with:
- `explain_prediction(model, X, idx)` → SHAP waterfall plot
- `global_importance(model, X)` → SHAP beeswarm plot
- `compare_importance(model, X, y)` → side-by-side MDI vs SHAP ranking
- Full docstrings, type hints, and error handling

**Artifact 2: Interactive Streamlit app** that lets the user:
1. Adjust `n_estimators` (1-500) and `max_features` (1-8) with sliders
2. See SHAP waterfall + beeswarm plots update with each parameter change
3. Compare RF vs Ridge vs GBR performance as hyperparameters change
4. Toggle between MDI, permutation, and SHAP importance rankings

### P.R.I.M.E. Prompt
Copy and paste this into Claude or ChatGPT:

In [ ]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — Co-Pilot required
# Copy the P.R.I.M.E. prompt above into Claude, then paste
# the generated code here. Run it and verify.
# -----------------------------------------------------------

# [Prep] Act as an expert Python Data Scientist specializing
# in SHAP explanations, interactive visualizations, and
# scikit-learn production workflows.
#
# [Request] I just completed a diagnosis-first lab where I
# compared Decision Trees, Ridge, Random Forests, and Gradient
# Boosting on California Housing data. I fixed evaluation bugs,
# diagnosed causal overclaiming from MDI, tuned hyperparameters
# with GridSearchCV, and generated SHAP waterfall + beeswarm
# plots. Now I need TWO artifacts:
#
# 1. A reusable `src/shap_utils.py` module with three functions:
#    - explain_prediction(model, X, idx) -> SHAP waterfall
#    - global_importance(model, X) -> SHAP beeswarm
#    - compare_importance(model, X, y) -> MDI vs SHAP side-by-side
#    Include type hints, docstrings, and error handling.
#
# 2. An interactive Plotly dashboard (or Streamlit app) with
#    ipywidgets sliders for n_estimators (1-500) and max_features
#    (1-8). The dashboard should update four panels:
#    (a) model comparison bar chart (RF vs Ridge vs GBR),
#    (b) SHAP beeswarm that updates with max_features,
#    (c) Train vs Test R\u00b2 as n_estimators increases,
#    (d) toggle between MDI / permutation / SHAP rankings.
#
# [Iterate] Use plotly.graph_objects, ipywidgets, shap, numpy,
# sklearn. Use the same variable names: X_train, X_test,
# y_train, y_test, data.feature_names. Do not use deprecated
# Plotly or SHAP functions.
#
# [Mechanism Check] Add inline comments explaining:
#   - How TreeExplainer differs from KernelExplainer
#   - Why SHAP values are additive (Shapley property)
#   - How ipywidgets observers trigger plot updates
#   - Why we re-fit inside the callback
#
# [Evaluate] Explain what the dashboard reveals about:
#   - The relationship between n_estimators, max_features,
#     and test performance
#   - Where MDI and SHAP rankings diverge and why
#   - The marginal value of additional trees beyond ~200

# PASTE AI-GENERATED CODE BELOW:


"""
shap_utils.py
=============
Reusable SHAP utility module for Lab 19 — Tree-Based Models & Random Forests
ECON 5200: Causal Machine Learning & Applied Analytics

Provides three functions:
    - explain_prediction()  : SHAP waterfall plot for a single observation
    - global_importance()   : SHAP beeswarm plot over the full dataset
    - compare_importance()  : Side-by-side MDI vs SHAP ranking bar chart

All functions use shap.TreeExplainer, which is exact and fast for tree ensembles.
For non-tree models, swap in shap.KernelExplainer (slower, approximate).
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from typing import Union

# Type alias for supported sklearn tree ensemble models
TreeModel = Union[RandomForestRegressor, GradientBoostingRegressor]


# ---------------------------------------------------------------------------
# Private helper: build or reuse a TreeExplainer + shap_values
# ---------------------------------------------------------------------------
def _get_shap_values(
    model: TreeModel,
    X: pd.DataFrame,
) -> tuple:
    """
    Build a shap.TreeExplainer and compute SHAP values.

    TreeExplainer uses the exact TreeSHAP algorithm — it exploits the
    tree structure to compute exact Shapley values in polynomial time,
    unlike KernelExplainer which is model-agnostic but uses Monte Carlo
    sampling and can be 100x slower.

    Parameters
    ----------
    model : fitted sklearn tree ensemble
    X     : feature matrix (DataFrame preserves feature names)

    Returns
    -------
    explainer   : shap.TreeExplainer
    shap_values : np.ndarray of shape (n_samples, n_features)
    """
    # check_additivity=False avoids a slow consistency check in some RF versions
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X, check_additivity=False)
    return explainer, shap_values


# ---------------------------------------------------------------------------
# 1. explain_prediction
# ---------------------------------------------------------------------------
def explain_prediction(
    model: TreeModel,
    X: pd.DataFrame,
    idx: int,
    title_suffix: str = "",
    save_path: str | None = None,
) -> None:
    """
    Generate a SHAP waterfall plot explaining a single prediction.

    A waterfall plot starts from the model's expected output (E[f(x)]),
    then shows each feature's additive contribution — positive (red) or
    negative (blue) — to arrive at f(x) for this observation.

    SHAP values are additive by the Shapley property: the sum of all
    SHAP values plus the base value exactly equals the model's output.
    This additivity comes from cooperative game theory — each feature
    receives its "fair share" of the prediction gap from the baseline.

    Parameters
    ----------
    model       : fitted sklearn tree ensemble (RandomForest or GBR)
    X           : feature matrix as a DataFrame (test or full set)
    idx         : integer row index of the observation to explain
    title_suffix: optional string appended to the plot title
    save_path   : if provided, saves the figure to this path (.png)

    Returns
    -------
    None  (displays or saves a matplotlib figure)

    Example
    -------
    >>> explain_prediction(best_rf, X_test, idx=0, title_suffix="High-value home")
    """
    if idx < 0 or idx >= len(X):
        raise IndexError(
            f"idx={idx} is out of bounds for X with {len(X)} rows."
        )

    explainer, shap_values = _get_shap_values(model, X)

    shap_explanation = shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X.iloc[idx].values,
        feature_names=list(X.columns),
    )

    title = f"SHAP Waterfall — Observation {idx}"
    if title_suffix:
        title += f" ({title_suffix})"

    plt.figure(figsize=(9, 5))
    shap.plots.waterfall(shap_explanation, show=False)
    plt.title(title, fontsize=12, pad=10)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    else:
        plt.show()
    plt.close()


# ---------------------------------------------------------------------------
# 2. global_importance
# ---------------------------------------------------------------------------
def global_importance(
    model: TreeModel,
    X: pd.DataFrame,
    max_display: int = 10,
    save_path: str | None = None,
) -> None:
    """
    Generate a SHAP beeswarm plot showing global feature importance.

    Each dot is one observation. The x-axis is the SHAP value (impact on
    prediction). Color encodes the raw feature value (red = high, blue = low).
    Features are sorted by mean |SHAP value|, giving a magnitude-ranked
    global importance without the cardinality bias of MDI.

    Parameters
    ----------
    model       : fitted sklearn tree ensemble
    X           : feature matrix as a DataFrame
    max_display : number of top features to display (default 10)
    save_path   : if provided, saves the figure to this path (.png)

    Returns
    -------
    None  (displays or saves a matplotlib figure)

    Example
    -------
    >>> global_importance(best_rf, X_test, max_display=8)
    """
    explainer, shap_values = _get_shap_values(model, X)

    shap_explanation = shap.Explanation(
        values=shap_values,
        base_values=explainer.expected_value,
        data=X.values,
        feature_names=list(X.columns),
    )

    plt.figure(figsize=(9, 6))
    shap.plots.beeswarm(shap_explanation, max_display=max_display, show=False)
    plt.title("SHAP Beeswarm — Global Feature Importance", fontsize=12, pad=10)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    else:
        plt.show()
    plt.close()


# ---------------------------------------------------------------------------
# 3. compare_importance
# ---------------------------------------------------------------------------
def compare_importance(
    model: TreeModel,
    X: pd.DataFrame,
    y: pd.Series | np.ndarray,
    n_repeats: int = 10,
    random_state: int = 42,
    save_path: str | None = None,
) -> pd.DataFrame:
    """
    Generate a side-by-side bar chart comparing MDI vs SHAP feature rankings.

    MDI (Mean Decrease in Impurity) is fast but biased toward high-cardinality
    features — continuous variables have more split opportunities, so they
    accumulate impurity reduction by chance.

    SHAP importance (mean |SHAP value|) is unbiased: it measures each feature's
    actual contribution to predictions on held-out observations, using the
    additive Shapley decomposition.

    Parameters
    ----------
    model        : fitted sklearn RandomForestRegressor (must have
                   feature_importances_ attribute; GBR also supported)
    X            : feature matrix as a DataFrame (ideally the test set)
    y            : target vector aligned with X
    n_repeats    : permutation importance repetitions (default 10)
    random_state : RNG seed for permutation importance
    save_path    : if provided, saves the figure to this path (.png)

    Returns
    -------
    importance_df : pd.DataFrame with columns [MDI, SHAP] sorted by SHAP rank

    Example
    -------
    >>> df = compare_importance(best_rf, X_test, y_test, save_path="figures/compare.png")
    >>> print(df)
    """
    feature_names = list(X.columns)

    # --- MDI (built-in to the model) ---
    if not hasattr(model, "feature_importances_"):
        raise AttributeError(
            "Model does not have feature_importances_. "
            "Use a tree-based ensemble (RandomForest, GBR)."
        )
    mdi = pd.Series(model.feature_importances_, index=feature_names)

    # --- SHAP mean |value| ---
    _, shap_values = _get_shap_values(model, X)
    shap_mean = pd.Series(
        np.abs(shap_values).mean(axis=0), index=feature_names
    )
    # Normalize to [0,1] for fair visual comparison
    shap_mean_norm = shap_mean / shap_mean.sum()

    # --- Build comparison DataFrame, sort by SHAP ---
    importance_df = pd.DataFrame(
        {"MDI": mdi, "SHAP": shap_mean_norm}
    ).sort_values("SHAP", ascending=True)  # ascending for horizontal bar

    # --- Plot ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    colors = {"MDI": "#4C72B0", "SHAP": "#DD8452"}

    for ax, col in zip(axes, ["MDI", "SHAP"]):
        bars = ax.barh(
            importance_df.index,
            importance_df[col],
            color=colors[col],
            edgecolor="white",
            linewidth=0.5,
        )
        ax.set_title(
            f"{'MDI (Mean Decrease Impurity)' if col == 'MDI' else 'SHAP (mean |value|)'}",
            fontsize=11,
        )
        ax.set_xlabel("Normalized Importance", fontsize=9)
        ax.spines[["top", "right"]].set_visible(False)
        # Annotate values
        for bar, val in zip(bars, importance_df[col]):
            ax.text(
                val + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center", fontsize=8,
            )

    fig.suptitle(
        "MDI vs SHAP Feature Importance\n"
        "(MDI is biased toward high-cardinality features; SHAP is unbiased)",
        fontsize=12, y=1.02,
    )
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    else:
        plt.show()
    plt.close()

    return importance_df.sort_values("SHAP", ascending=False)


# ---------------------------------------------------------------------------
# __main__ demo — run with: python src/shap_utils.py
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    """
    Demonstration block: trains a small RF on California Housing and
    exercises all three functions. Run from repo root:
        python src/shap_utils.py
    """
    from sklearn.datasets import fetch_california_housing
    from sklearn.model_selection import train_test_split
    import os

    print("Loading California Housing data...")
    data = fetch_california_housing()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = data.target
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print("Training RandomForestRegressor (n_estimators=100)...")
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    os.makedirs("figures", exist_ok=True)

    print("\n[1/3] Waterfall plot — observation 0 (high-value home)...")
    explain_prediction(
        model, X_test, idx=0,
        title_suffix="demo",
        save_path="figures/shap_waterfall_demo.png",
    )

    print("[2/3] Beeswarm plot — global importance...")
    global_importance(
        model, X_test,
        save_path="figures/shap_beeswarm_demo.png",
    )

    print("[3/3] MDI vs SHAP comparison...")
    df = compare_importance(
        model, X_test, y_test,
        save_path="figures/compare_importance_demo.png",
    )
    print("\nImportance ranking (sorted by SHAP):")
    print(df.round(4).to_string())

    print("\n✅ shap_utils.py demo complete. Check figures/ directory.")









Loading California Housing data...
Training RandomForestRegressor (n_estimators=100)...

[1/3] Waterfall plot — observation 0 (high-value home)...


---
## Digital Portfolio: Institutional Signaling

### Generate Your Professional README
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

In [ ]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — README generation (no code, just docs)
# -----------------------------------------------------------

# PASTE THIS PROMPT INTO CLAUDE:
#
# "I need help writing a project description for my data science lab.
# **Important Rule:** Do NOT generate any Python code for me.
#
# **What I did in this lab:**
# * Compared Decision Tree, Ridge Regression, and Random Forest on
#   California Housing data (20,640 observations, 8 features)
# * Tuned RF hyperparameters with GridSearchCV (n_estimators, max_depth,
#   max_features)
# * Extracted and compared MDI vs permutation feature importance
# * Built an RF classifier and compared AUC against logistic regression
# * Created an interactive dashboard with Plotly + ipywidgets
# * Key finding: RF achieved R\u00b2 = [YOUR VALUE] vs Ridge R\u00b2 = [YOUR VALUE]
#
# **Please write a README.md entry including:**
# 1. Project Title: Tree-Based Models \u2014 Random Forests
# 2. Objective: A professional one-sentence summary
# 3. Methodology: Bullet points of technical steps
# 4. Key Findings: Summary of results
# Make this sound like a professional tech economist wrote it."

### Push to GitHub

```bash
cd econ-lab-19-random-forests
git add notebooks/ figures/ README.md verification-log.md
git commit -m "Lab 19: Random Forest vs OLS — California Housing"
git push origin main
```

Submit your GitHub repo link on Canvas.